In [ ]:
# NSE F&O Bullish Screener for Intraday Longs
# This script screens NSE F&O stocks for bullish setups based on volume surge, low volatility,
# technical indicators (RSI >50, Close >50-day EMA, MACD bullish), OI buildup from Bhavcopy,
# and PCR (Put-Call Ratio <0.9 for bullish bias).
# Requirements: pip install pandas pandas_ta numpy requests
# Note: For OI/PCR from Bhavcopy:
# 1. Download latest F&O-UDiFF Common Bhavcopy Final.zip from https://www.nseindia.com/all-reports-derivatives
# 2. Unzip to get BhavCopy_NSE_FO_0_0_0_YYYYMMDD_F_0000.csv (e.g., for Oct 26, 2025: BhavCopy_NSE_FO_0_0_0_20251026_F_0000.csv)
# 3. Place the CSV in the same directory and update file_path below.
# Symbols are dynamically read from FUTSTK in Bhavcopy (unique SYMBOLs).
# Historical data now fetched directly from NSE API instead of yfinance, with enhanced error handling.

import json
import pandas as pd
import pandas_ta as ta
import numpy as np
import requests
from io import StringIO
from datetime import datetime, timedelta
import time
import random
import zipfile, io

# Parameters
end_date = datetime.now()
start_date = end_date - timedelta(days=100)  # Enough for 50-day EMA
volume_multiplier = 1.5
rsi_threshold = 50
atr_percent_threshold = 5.0
oi_pct_threshold = 10.0  # OI increase >10%
pcr_threshold = 0.9  # PCR <0.9 (more call OI, bullish)
nifty_support = 25000  # Manual check; fetch Nifty if needed
max_retries = 3  # For API retries
retry_delay = 2  # Initial delay in seconds

# Bhavcopy file path (update with actual unzipped CSV name)
bhavcopy_file = 'BhavCopy_NSE_FO_0_0_0_20251026_F_0000.csv'  # Example for Oct 26, 2025

def download_and_load_fo(date: datetime) -> pd.DataFrame:
    """Download NSE FO bhav-copy for a date and return DF or raise."""
 # Format the date
    date_str = date.strftime('%Y%m%d')  # YYYYMMDD
    filename = f'BhavCopy_NSE_FO_0_0_0_{date_str}_F_0000.csv.zip'
    url = f'https://nsearchives.nseindia.com/content/fo/{filename}'

    print(f"Fetching {url}")
    
    # Use headers to simulate a browser
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Referer': 'https://www.nseindia.com/'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            z = zipfile.ZipFile(io.BytesIO(response.content))
            csv_name = z.namelist()[0]
            with z.open(csv_name) as f:
                df = pd.read_csv(f)

            df.rename(columns=
                                {'FinInstrmTp': 'INSTRUMENT'
                                 , 'TckrSymb': 'SYMBOL'
                                 , 'XpryDt': 'EXPIRY_DT'
                                 , 'StrkPric': 'STRIKE_PR'
                                 , 'OptnTp': 'OPTION_TYP'
                                 , 'OpnPric': 'OPEN'
                                 , 'HghPric': 'HIGH'
                                 , 'LwPric': 'LOW'
                                 , 'ClsPric': 'CLOSE'
                                 , 'LastPric': 'SETTLE_PR'
                                 , 'TtlTradgVol': 'CONTRACTS'
                                 , 'TtlTrfVal': 'VAL_INLAKH'
                                 , 'OpnIntrst': 'OPEN_INT'
                                 , 'ChngInOpnIntrst': 'CHG_IN_OI'
                                 , 'TradDt': 'TIMESTAMP'
                                 }, inplace=True)    
            # Standardise column names
            #df.columns = df.columns.str.lower()
            # df.to_csv(NIFTY_CSV, index=False, mode='w', header=True)            
            
            return df
        else:
            print(f"Failed to download. HTTP Status Code: {response.status_code}")
            print("Possible reasons: Non-trading day, wrong URL format, or file not available yet.")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error occurred: {e}")
        return pd.DataFrame()


def parse_bhavcopy():
    """Parse NSE F&O Bhavcopy CSV for OI, CHG_IN_OI, and PCR per symbol (futures + options)."""
    try:
        #today = datetime.now().date()
        today = datetime(2025, 10, 27)  # Example date for testing
        df = download_and_load_fo(today)
        
        # df.to_csv(BHAVCOPY_CSV, index=False, mode='w')
        # df = pd.read_csv(file_path)

        # Filter for equity futures and options (FUTSTK and OPTSTK)
        eq_data = df[df['INSTRUMENT'].isin(['STF', 'STO'])]  # Assuming 'INSTRUMENT' column
        
        # Get unique F&O symbols from FUTSTK
        fut_df = eq_data[eq_data['INSTRUMENT'] == 'STF']
        symbols = fut_df['SYMBOL'].unique()

        # For each symbol, calculate OI change for futures and PCR for options
        results = []
        for sym in symbols:
            # Futures OI for this symbol
            fut_sym = fut_df[fut_df['SYMBOL'] == sym]
            if fut_sym.empty:
                continue
            open_int_fut = fut_sym['OPEN_INT'].sum()
            chg_oi_fut = fut_sym['CHG_IN_OI'].sum()
            pct_chg_oi = np.where(
                (open_int_fut - chg_oi_fut) > 0,
                (chg_oi_fut / (open_int_fut - chg_oi_fut)) * 100,
                0
            )
            # Ensure pct_chg_oi is a scalar float for rounding
            if isinstance(pct_chg_oi, np.ndarray):
                pct_chg_oi = float(pct_chg_oi.item())
            
            # Options OI for this symbol (CE and PE)
            opt_sym = eq_data[(eq_data['INSTRUMENT'] == 'STO') & (eq_data['SYMBOL'] == sym)]
            ce_oi = opt_sym[opt_sym['OPTION_TYP'] == 'CE']['OPEN_INT'].sum() if 'OPTION_TYP' in opt_sym.columns else 0
            pe_oi = opt_sym[opt_sym['OPTION_TYP'] == 'PE']['OPEN_INT'].sum() if 'OPTION_TYP' in opt_sym.columns else 0
            pcr = pe_oi / ce_oi if ce_oi > 0 else 1.0  # Default 1.0 if no CE
            
            results.append({
                'SYMBOL': sym.strip().upper(),
                'OPEN_INT_FUT': int(open_int_fut),
                'CHG_IN_OI_FUT': int(chg_oi_fut),
                'PCT_CHG_OI': round(pct_chg_oi, 2),
                'CE_OI': int(ce_oi),
                'PE_OI': int(pe_oi),
                'PCR': round(pcr, 2)
            })
        
        oi_summary = pd.DataFrame(results)
        return oi_summary, list(oi_summary['SYMBOL'])
    except Exception as e:
        print(f"Error parsing Bhavcopy: {e}")        
        return pd.DataFrame(columns=['SYMBOL', 'OPEN_INT_FUT', 'CHG_IN_OI_FUT', 'PCT_CHG_OI', 'CE_OI', 'PE_OI', 'PCR']), []

def fetch_data(symbol, retries=max_retries):
    """Fetch historical data directly from NSE API with retry logic and error handling."""
    sym = symbol.replace('.NS', '').replace('&', '%26')
    from_date = start_date.strftime('%d-%m-%Y')
    to_date = end_date.strftime('%d-%m-%Y')
    url = f"https://www.nseindia.com/api/historicalOR/generateSecurityWiseHistoricalData?from={from_date}&to={to_date}&symbol={sym}&type=priceVolumeDeliverable&series=EQ"
    

    headers = {
        "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "accept-language": "en-US,en;q=0.9,en-IN;q=0.8,en-GB;q=0.7",
        "cache-control": "max-age=0",
        "priority": "u=0, i",
        "sec-ch-ua": '"Microsoft Edge";v="129", "Not=A?Brand";v="8", "Chromium";v="129"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"Windows"',
        "sec-fetch-dest": "document",
        "sec-fetch-mode": "navigate",
        "sec-fetch-site": "none",
        "sec-fetch-user": "?1",
        "upgrade-insecure-requests": "1",
        "Referer": "https://www.nseindia.com/report-detail/eq_security",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36 Edg/129.0.0.0"
    }
    
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()  # Raises HTTPError for bad status codes
            
            data = json.dumps(response.json())
            parsed_data = json.loads(data)
            df = pd.DataFrame(parsed_data['data'])
            
            df.rename(columns=
                        {'CH_SYMBOL': 'SYMBOL'
                        , 'mTIMESTAMP': 'Date'
                        , 'CH_OPENING_PRICE': 'Open'
                        , 'CH_TRADE_HIGH_PRICE': 'High'
                        , 'CH_TRADE_LOW_PRICE': 'Low'
                        , 'CH_CLOSING_PRICE': 'Close'
                        , 'CH_TOT_TRADED_QTY': 'Volume'
                        , 'VWAP': 'VWAP'}, inplace=True) 
            
            df.set_index('Date', inplace=True)
            df.sort_index(inplace=True)
            if len(df) < 50:
                print(f"Warning: Insufficient data ({len(df)} rows) for {symbol}")
                return None
            
            return df
        
        except requests.exceptions.HTTPError as e:
            status_code = e.response.status_code if e.response else None
            if status_code == 403:
                print(f"Access forbidden (403) for {symbol}. Possible cookie/session issue. Skipping.")
            elif status_code == 429:
                print(f"Rate limit (429) hit for {symbol}. Retrying in {retry_delay * (2 ** attempt)}s...")
                time.sleep(retry_delay * (2 ** attempt) + random.uniform(0, 1))
                continue
            else:
                print(f"HTTP error {status_code} for {symbol}: {e}")
                if attempt == retries - 1:
                    return None
                time.sleep(retry_delay * (2 ** attempt) + random.uniform(0, 1))
                continue
        
        except (requests.exceptions.RequestException, ValueError, pd.errors.EmptyDataError) as e:
            print(f"Request/Parse error for {symbol} (attempt {attempt + 1}/{retries}): {e}")
            if attempt == retries - 1:
                print(f"Max retries exceeded for {symbol}. Skipping.")
                return None
            time.sleep(retry_delay * (2 ** attempt) + random.uniform(0, 1))
            continue
    
    return None

def calculate_indicators(data):
    """Calculate technical indicators using pandas_ta."""
    # RSI(14)
    data['RSI'] = ta.rsi(data['Close'], length=14)
    
    # 50-day EMA
    data['EMA50'] = ta.ema(data['Close'], length=50)
    
    # MACD (12,26,9)
    macd = ta.macd(data['Close'])
    data = pd.concat([data, macd], axis=1)
    
    # ATR(14)
    data['ATR'] = ta.atr(data['High'], data['Low'], data['Close'], length=14)
    
    # 20-day avg volume
    data['AvgVol20'] = data['Volume'].rolling(window=20).mean()
    
    return data

def is_bullish_setup(data, symbol_clean, oi_df):
    """Check bullish criteria on latest data, including OI from Bhavcopy and PCR."""
    if len(data) < 50:  # Need enough data
        return False
    
    latest = data.iloc[-1]
    prev = data.iloc[-2]
    
    # Volume surge: Today's volume > 1.5x 20-day avg
    volume_surge = latest['Volume'] > (volume_multiplier * latest['AvgVol20'])
    
    # Low volatility: ATR < 5% of close
    low_vol = (latest['ATR'] / latest['Close']) * 100 < atr_percent_threshold
    
    # RSI > 50
    rsi_bull = latest['RSI'] > rsi_threshold
    
    # Close > 50-day EMA
    ema_bull = latest['Close'] > latest['EMA50']
    
    # MACD bullish crossover: MACD line > signal (or recent cross)
    macd_bull = latest['MACD_12_26_9'] > latest['MACDs_12_26_9'] and \
                prev['MACD_12_26_9'] <= prev['MACDs_12_26_9']
    
    # Price filter: >100 INR (approx, using close)
    price_filter = latest['Close'] > 100
    
    # OI buildup: % change >10% (from Bhavcopy futures)
    oi_row = oi_df[oi_df['SYMBOL'] == symbol_clean]
    oi_bull = False
    pcr_bull = False
    if not oi_row.empty:
        pct_chg_oi = oi_row['PCT_CHG_OI'].iloc[0]
        oi_bull = pct_chg_oi > oi_pct_threshold
        pcr = oi_row['PCR'].iloc[0]
        pcr_bull = pcr < pcr_threshold
    else:
        oi_bull = True  # Skip if no OI data
        pcr_bull = True  # Skip PCR if no data
    
    return all([volume_surge, low_vol, rsi_bull, ema_bull, macd_bull, price_filter, oi_bull, pcr_bull])

def screen_stocks(oi_df, fo_symbols_from_bhav):
    """Run the screener with dynamic symbols from Bhavcopy."""
    results = []
    
    # Use symbols from Bhavcopy if available, else fallback to hardcoded
    symbols_to_screen = fo_symbols_from_bhav
    # symbols_to_screen = [fo_symbols_from_bhav[0]]
    # symbols_to_screen = [sym + '.NS' for sym in fo_symbols_from_bhav] if fo_symbols_from_bhav else fo_symbols
    
    for symbol in symbols_to_screen:
        symbol_clean = symbol.replace('.NS', '').upper()
        print(f"Screening {symbol}...")
        data = fetch_data(symbol)
        if data is None:
            continue
        
        print('Data fetched, calculating indicators...')

        data = calculate_indicators(data)
        if is_bullish_setup(data, symbol_clean, oi_df):
            latest = data.iloc[-1]
            oi_row = oi_df[oi_df['SYMBOL'] == symbol_clean]
            oi_info = oi_row.iloc[0] if not oi_row.empty else {'OPEN_INT_FUT': 0, 'PCT_CHG_OI': 0, 'PCR': 1.0}
            results.append({
                'Symbol': symbol.replace('.NS', ''),
                'Close': round(latest['Close'], 2),
                'RSI': round(latest['RSI'], 2),
                'Volume Surge': round(latest['Volume'] / latest['AvgVol20'], 2),
                'ATR %': round((latest['ATR'] / latest['Close']) * 100, 2),
                'OI': int(oi_info['OPEN_INT_FUT']) if oi_info['OPEN_INT_FUT'] > 0 else 0,
                'OI % Chg': round(oi_info['PCT_CHG_OI'], 2),
                'PCR': round(oi_info['PCR'], 2)
            })
    
    df = pd.DataFrame(results)
    if not df.empty:
        print("\nBullish F&O Stocks for Long:")
        print(df.to_string(index=False))
    else:
        print("\nNo stocks meet the criteria today.")
    
    return df


# Run the screener
if __name__ == "__main__":
    # Parse Bhavcopy for OI, PCR, and symbols
    oi_df, fo_symbols_from_bhav = parse_bhavcopy()
    print(f"Parsed data for {len(oi_df)} symbols from Bhavcopy.")
    if not oi_df.empty:
        #print(oi_df.head())
        print(f"Dynamic F&O symbols: {len(fo_symbols_from_bhav)}")
    
    # Optional: Check Nifty sentiment (using yfinance for simplicity, as NSE index endpoint differs)
    import yfinance as yf
    nifty = yf.Ticker('^NSEI').history(period='1d')
    if not nifty.empty and nifty['Close'].iloc[-1] > nifty_support:
        print(f"Nifty above support ({nifty_support}): Bullish bias confirmed.")
    else:
        print("Nifty below support: Avoid longs.")
    
    screen_stocks(oi_df, fo_symbols_from_bhav)

# Notes:
# - Enhanced error handling in fetch_data: Retries on HTTP 429/403 with exponential backoff and jitter.
# - Validates response content (CSV vs error JSON), required columns, and data length.
# - PCR calculated as total PE OI / total CE OI across all strikes/expiries for the symbol.
# - Assumes 'INSTRUMENT', 'SYMBOL', 'OPEN_INT', 'CHG_IN_OI', 'OPTION_TYP' columns in CSV; adjust if column names differ.
# - For news catalyst: Integrate Alpha Vantage or manual check.
# - Run daily pre-market after downloading/unzipping Bhavcopy. Backtest by changing end_date.
# - If NSE API blocks requests persistently, consider adding cookies or fallback to yfinance.
# - Example output: Filters to stocks with low PCR, now dynamic symbols, direct from NSE.